**Проделаем обработку тестовых данных**

In [ ]:
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
from sklearn import linear_model
from sklearn import preprocessing
from sklearn import model_selection
from sklearn import tree
from sklearn import ensemble
from sklearn import metrics
from sklearn import cluster
from sklearn import feature_selection

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/data/Project5_test_data.zip

Archive:  /content/drive/MyDrive/data/Project5_test_data.zip
  inflating: Project5_test_data.csv  


Прочитаем наш файл с исходными данными:

In [ ]:
taxi_data = pd.read_csv("/content/drive/MyDrive/data/Project5_test_data.zip")
print('Train data shape: {}'.format(taxi_data.shape))
taxi_data.head()

Train data shape: (625134, 9)


,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N


In [ ]:
taxi_data['pickup_datetime'] = pd.to_datetime(taxi_data['pickup_datetime'], format='%Y-%m-%d %H:%M:%S')
taxi_data.describe()

,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
count,625134.000000,625134,625134.000000,625134.000000,625134.000000,625134.000000,625134.000000
mean,1.534884,2016-04-01 13:27:01.567467264,1.661765,-73.973614,40.750927,-73.973458,40.751816
min,1.000000,2016-01-01 00:00:22,0.000000,-121.933128,37.389587,-121.933327,36.601322
25%,1.000000,2016-02-17 19:44:19,1.000000,-73.991852,40.737392,-73.991318,40.736000
50%,2.000000,2016-04-01 20:01:43,1.000000,-73.981743,40.754093,-73.979774,40.754543
75%,2.000000,2016-05-15 10:07:52.750000128,2.000000,-73.967400,40.768394,-73.963013,40.769852
max,2.000000,2016-06-30 23:59:58,9.000000,-69.248917,42.814938,-67.496796,48.857597
std,0.498782,NaN,1.311293,0.073389,0.029848,0.072565,0.035824


In [ ]:
taxi_data.info()
print('Всего пропусков: ', taxi_data.isna().sum().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625134 entries, 0 to 625133
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id                  625134 non-null  object        
 1   vendor_id           625134 non-null  int64         
 2   pickup_datetime     625134 non-null  datetime64[ns]
 3   passenger_count     625134 non-null  int64         
 4   pickup_longitude    625134 non-null  float64       
 5   pickup_latitude     625134 non-null  float64       
 6   dropoff_longitude   625134 non-null  float64       
 7   dropoff_latitude    625134 non-null  float64       
 8   store_and_fwd_flag  625134 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(2)
memory usage: 42.9+ MB
Всего пропусков:  0


In [ ]:
def add_datatime_features(df):
  df['pickup_data'] = df['pickup_datetime'].dt.date
  df['pickup_hour'] = df['pickup_datetime'].dt.hour
  df['pickup_day_of_week'] = df['pickup_datetime'].dt.day_name()
  return df

taxi_data = add_datatime_features(taxi_data)
taxi_data.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,pickup_hour,pickup_day_of_week
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,23,Thursday
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,23,Thursday
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,23,Thursday
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,23,Thursday
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,23,Thursday


In [ ]:
gdown.download(url='https://drive.google.com/file/d/1sspPYZ1cVe7OG4DnTTZEP_oqMtMZJTLm/view?usp=sharing', output='holiday_data.csv', fuzzy=True)

Downloading...
From: https://drive.google.com/uc?id=1sspPYZ1cVe7OG4DnTTZEP_oqMtMZJTLm
To: /content/holiday_data.csv
100%|██████████| 508/508 [00:00<00:00, 1.30MB/s]


'holiday_data.csv'

In [ ]:
holiday_data = pd.read_csv('/content/drive/MyDrive/data/holiday_data.csv', sep=';')
holiday_data.head()

,day,date,holiday
0,Friday,2016-01-01,New Years Day
1,Monday,2016-01-18,Martin Luther King Jr. Day
2,Friday,2016-02-12,Lincoln's Birthday
3,Monday,2016-02-15,Presidents' Day
4,Sunday,2016-05-08,Mother's Day


In [ ]:
holiday_data['date'] = pd.to_datetime(holiday_data['date'], format='%Y-%m-%d')
holiday_data.head()

,day,date,holiday
0,Friday,2016-01-01,New Years Day
1,Monday,2016-01-18,Martin Luther King Jr. Day
2,Friday,2016-02-12,Lincoln's Birthday
3,Monday,2016-02-15,Presidents' Day
4,Sunday,2016-05-08,Mother's Day


In [ ]:
taxi_data['pickup_data'] = pd.to_datetime(taxi_data['pickup_data'], format='%Y-%m-%d')

In [ ]:
def add_holiday_features(arg):
  if arg != 0:
    arg = 1
  return arg

taxi_data = taxi_data.merge(holiday_data, left_on='pickup_data', how='left', right_on='date')
taxi_data['holiday'] = taxi_data['holiday'].fillna(0)
taxi_data['holiday'] = taxi_data['holiday'].apply(add_holiday_features)
taxi_data.head(10)

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,pickup_hour,pickup_day_of_week,day,date,holiday
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,23,Thursday,NaN,NaT,0
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,23,Thursday,NaN,NaT,0
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,23,Thursday,NaN,NaT,0
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,23,Thursday,NaN,NaT,0
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,23,Thursday,NaN,NaT,0
5,id0668992,1,2016-06-30 23:59:30,1,-73.991302,40.749798,-73.980515,40.786549,N,2016-06-30,23,Thursday,NaN,NaT,0
6,id1765014,1,2016-06-30 23:59:15,1,-73.978310,40.741550,-73.952072,40.717003,N,2016-06-30,23,Thursday,NaN,NaT,0
7,id0898117,1,2016-06-30 23:59:09,2,-74.012711,40.701527,-73.986481,40.719509,N,2016-06-30,23,Thursday,NaN,NaT,0
8,id3905224,2,2016-06-30 23:58:55,2,-73.992332,40.730511,-73.875618,40.875214,N,2016-06-30,23,Thursday,NaN,NaT,0
9,id1543102,2,2016-06-30 23:58:46,1,-73.993179,40.748760,-73.979309,40.761311,N,2016-06-30,23,Thursday,NaN,NaT,0


In [ ]:
!unzip /content/drive/MyDrive/data/Project5_osrm_data_test.zip

Archive:  /content/drive/MyDrive/data/Project5_osrm_data_test.zip
  inflating: Project5_osrm_data_test.csv  


In [ ]:
osrm_data = pd.read_csv('/content/Project5_osrm_data_test.csv')
osrm_data.head()

,id,starting_street,end_street,total_distance,total_travel_time,number_of_steps,street_for_each_step,distance_per_step,travel_time_per_step,step_maneuvers,step_direction,step_location_list
0,id0771704,6th Avenue,10th Avenue,1497.1,200.2,7,6th Avenue|West 17th Street|9th Avenue|9th Ave...,188.7|825.1|96.4|58.6|267.5|60.7|0,32.4|103.7|12.7|10.7|34|6.7|0,depart|turn|turn|continue|turn|turn|arrive,left|left|left|slight right|right|right|arrive,"-73.996527,40.737786|-73.995446,40.739272|-74...."
1,id3274209,5th Avenue,5th Avenue,1427.1,141.5,2,5th Avenue|5th Avenue,1427.1|0,141.5|0,depart|arrive,none|arrive,"-73.976974,40.75885|-73.98516,40.747618"
2,id2756455,East 18th Street,Park Avenue,2312.3,324.6,9,East 18th Street|Irving Place|East 19th Street...,19.3|74.1|148.9|632.1|163.9|1111.5|144.7|17.7|0,9.6|15.6|20.7|92.1|24.4|136.2|24.9|1.1|0,depart|turn|turn|turn|turn|turn|turn|turn|arrive,left|left|left|right|left|right|right|right|ar...,"-73.987161,40.736551|-73.986961,40.736466|-73...."
3,id3684027,Madison Avenue,6th Avenue,931.8,84.2,4,Madison Avenue|East 49th Street|6th Avenue|6th...,199.3|466.6|265.8|0,24.4|36.5|23.3|0,depart|turn|turn|arrive,right|left|right|arrive,"-73.977191,40.755664|-73.976045,40.757232|-73...."
4,id3101285,Madison Avenue,West 83rd Street,2501.7,294.7,8,Madison Avenue|East 91st Street|5th Avenue|86t...,31|154.9|491|907.6|272.1|248|397.2|0,9.4|29.7|46|103.6|25.1|21.8|59.1|0,depart|turn|turn|turn|new name|turn|turn|arrive,right|left|left|right|straight|left|right|arrive,"-73.956667,40.783797|-73.956489,40.784041|-73...."


In [ ]:
osrm_data1 = osrm_data.loc[:, ['id', 'total_distance', 'total_travel_time', 'number_of_steps']]
osrm_data1.head()

,id,total_distance,total_travel_time,number_of_steps
0,id0771704,1497.1,200.2,7
1,id3274209,1427.1,141.5,2
2,id2756455,2312.3,324.6,9
3,id3684027,931.8,84.2,4
4,id3101285,2501.7,294.7,8


In [ ]:
taxi_data = taxi_data.merge(osrm_data1, how='left')
taxi_data.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,pickup_hour,pickup_day_of_week,day,date,holiday,total_distance,total_travel_time,number_of_steps
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,23,Thursday,NaN,NaT,0,3795.9,424.6,4
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,23,Thursday,NaN,NaT,0,2904.5,200.0,4
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,23,Thursday,NaN,NaT,0,1499.5,193.2,4
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,23,Thursday,NaN,NaT,0,7023.9,494.8,11
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,23,Thursday,NaN,NaT,0,1108.2,103.2,4


In [ ]:
def get_haversine_distance(lat1, lng1, lat2, lng2):
    # переводим углы в радианы
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    # радиус земли в километрах
    EARTH_RADIUS = 6371
    # считаем кратчайшее расстояние h по формуле Хаверсина
    lat_delta = lat2 - lat1
    lng_delta = lng2 - lng1
    d = np.sin(lat_delta * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng_delta * 0.5) ** 2
    h = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(d))
    return h

def get_angle_direction(lat1, lng1, lat2, lng2):
    # переводим углы в радианы
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    # считаем угол направления движения alpha по формуле угла пеленга
    lng_delta_rad = lng2 - lng1
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    alpha = np.degrees(np.arctan2(y, x))
    return alpha

In [ ]:
def add_geographical_features(df):
  df['haversine_distance'] = get_haversine_distance(df['pickup_latitude'],
                                                    df['pickup_longitude'],
                                                    df['dropoff_latitude'],
                                                    df['dropoff_longitude'])
  df['direction'] = get_angle_direction(df['pickup_latitude'],
                                        df['pickup_longitude'],
                                        df['dropoff_latitude'],
                                        df['dropoff_longitude'])
  return df

taxi_data = add_geographical_features(taxi_data)
taxi_data.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,pickup_hour,pickup_day_of_week,day,date,holiday,total_distance,total_travel_time,number_of_steps,haversine_distance,direction
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,23,Thursday,NaN,NaT,0,3795.9,424.6,4,2.746426,-3.595224
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,23,Thursday,NaN,NaT,0,2904.5,200.0,4,2.759239,172.278835
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,23,Thursday,NaN,NaT,0,1499.5,193.2,4,1.306155,133.326248
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,23,Thursday,NaN,NaT,0,7023.9,494.8,11,5.269088,-150.956833
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,23,Thursday,NaN,NaT,0,1108.2,103.2,4,0.960842,130.260381


In [ ]:
import joblib

kmeans = joblib.load('/content/drive/MyDrive/models/kmeansPP2.pkl')

In [ ]:
def add_cluster_features(df, model):
  coords = np.hstack((df[['pickup_latitude', 'pickup_longitude']],
                    df[['dropoff_latitude', 'dropoff_longitude']]))
  df['geo_cluster'] = model.predict(coords)
  return df

taxi_data = add_cluster_features(taxi_data, kmeans)
taxi_data.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,...,pickup_day_of_week,day,date,holiday,total_distance,total_travel_time,number_of_steps,haversine_distance,direction,geo_cluster
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,...,Thursday,NaN,NaT,0,3795.9,424.6,4,2.746426,-3.595224,1
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,...,Thursday,NaN,NaT,0,2904.5,200.0,4,2.759239,172.278835,6
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,...,Thursday,NaN,NaT,0,1499.5,193.2,4,1.306155,133.326248,6
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,...,Thursday,NaN,NaT,0,7023.9,494.8,11,5.269088,-150.956833,1
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,...,Thursday,NaN,NaT,0,1108.2,103.2,4,0.960842,130.260381,1


In [ ]:
taxi_data.groupby(by='geo_cluster')['id'].count().sort_values()

,id
geo_cluster,
2,1
9,8
7,6613
5,13888
4,17121
8,19439
6,145699
0,167665
1,254700


In [ ]:
gdown.download(url='https://drive.google.com/file/d/1SLIS4sULxi2SAjmqoeSnbCu6bKEN-zPm/view?usp=sharing', output='weather_data.zip', fuzzy=True)

Downloading...
From: https://drive.google.com/uc?id=1SLIS4sULxi2SAjmqoeSnbCu6bKEN-zPm
To: /content/weather_data.zip
100%|██████████| 134k/134k [00:00<00:00, 28.4MB/s]


'weather_data.zip'

In [ ]:
!unzip /content/weather_data.zip

Archive:  /content/weather_data.zip
  inflating: weather_data.csv        


In [ ]:
weather_data = pd.read_csv('weather_data.csv')
weather_data.head()

,time,temperature,windchill,heat index,humidity,pressure,dew Point,visibility,wind dir,wind speed,gust speed,precip,events,conditions,date,hour
0,2015-12-31 02:00:00,7.8,7.1,NaN,0.89,1017.0,6.1,8.0,NNE,5.6,0.0,0.8,NaN,Overcast,2015-12-31,2
1,2015-12-31 03:00:00,7.2,5.9,NaN,0.90,1016.5,5.6,12.9,Variable,7.4,0.0,0.3,NaN,Overcast,2015-12-31,3
2,2015-12-31 04:00:00,7.2,NaN,NaN,0.90,1016.7,5.6,12.9,Calm,0.0,0.0,0.0,NaN,Overcast,2015-12-31,4
3,2015-12-31 05:00:00,7.2,5.9,NaN,0.86,1015.9,5.0,14.5,NW,7.4,0.0,0.0,NaN,Overcast,2015-12-31,5
4,2015-12-31 06:00:00,7.2,6.4,NaN,0.90,1016.2,5.6,11.3,West,5.6,0.0,0.0,NaN,Overcast,2015-12-31,6


In [ ]:
weather_data['time'] = pd.to_datetime(weather_data['time'], format='%Y-%m-%d %H:%M:%S')
weather_data.head()

,time,temperature,windchill,heat index,humidity,pressure,dew Point,visibility,wind dir,wind speed,gust speed,precip,events,conditions,date,hour
0,2015-12-31 02:00:00,7.8,7.1,NaN,0.89,1017.0,6.1,8.0,NNE,5.6,0.0,0.8,NaN,Overcast,2015-12-31,2
1,2015-12-31 03:00:00,7.2,5.9,NaN,0.90,1016.5,5.6,12.9,Variable,7.4,0.0,0.3,NaN,Overcast,2015-12-31,3
2,2015-12-31 04:00:00,7.2,NaN,NaN,0.90,1016.7,5.6,12.9,Calm,0.0,0.0,0.0,NaN,Overcast,2015-12-31,4
3,2015-12-31 05:00:00,7.2,5.9,NaN,0.86,1015.9,5.0,14.5,NW,7.4,0.0,0.0,NaN,Overcast,2015-12-31,5
4,2015-12-31 06:00:00,7.2,6.4,NaN,0.90,1016.2,5.6,11.3,West,5.6,0.0,0.0,NaN,Overcast,2015-12-31,6


In [ ]:
weather_data['date'] = weather_data['time'].dt.date
weather_data.head()

,time,temperature,windchill,heat index,humidity,pressure,dew Point,visibility,wind dir,wind speed,gust speed,precip,events,conditions,date,hour
0,2015-12-31 02:00:00,7.8,7.1,NaN,0.89,1017.0,6.1,8.0,NNE,5.6,0.0,0.8,NaN,Overcast,2015-12-31,2
1,2015-12-31 03:00:00,7.2,5.9,NaN,0.90,1016.5,5.6,12.9,Variable,7.4,0.0,0.3,NaN,Overcast,2015-12-31,3
2,2015-12-31 04:00:00,7.2,NaN,NaN,0.90,1016.7,5.6,12.9,Calm,0.0,0.0,0.0,NaN,Overcast,2015-12-31,4
3,2015-12-31 05:00:00,7.2,5.9,NaN,0.86,1015.9,5.0,14.5,NW,7.4,0.0,0.0,NaN,Overcast,2015-12-31,5
4,2015-12-31 06:00:00,7.2,6.4,NaN,0.90,1016.2,5.6,11.3,West,5.6,0.0,0.0,NaN,Overcast,2015-12-31,6


In [ ]:
weather_data['date'] = pd.to_datetime(weather_data['date'], format='%Y-%m-%d')
weather_data = weather_data[['date', 'hour', 'temperature', 'visibility', 'wind speed', 'precip', 'events']]
weather_data.head()

,date,hour,temperature,visibility,wind speed,precip,events
0,2015-12-31,2,7.8,8.0,5.6,0.8,NaN
1,2015-12-31,3,7.2,12.9,7.4,0.3,NaN
2,2015-12-31,4,7.2,12.9,0.0,0.0,NaN
3,2015-12-31,5,7.2,14.5,7.4,0.0,NaN
4,2015-12-31,6,7.2,11.3,5.6,0.0,NaN


In [ ]:
taxi_data = taxi_data.merge(weather_data, how='left', left_on=['pickup_data', 'pickup_hour'], right_on=['date', 'hour'])

In [ ]:
def fill_null_weather_data(df):
  for col in ['temperature', 'visibility', 'wind speed', 'precip']:
    df[col] = df[col].fillna(df.groupby('pickup_data')[col].transform('median'))
  df['events'].fillna('None', inplace=True)
  for col in 'total_distance', 'total_travel_time', 'number_of_steps':
    df[col].fillna(df[col].median(), inplace=True)
  return df

taxi_data = fill_null_weather_data(taxi_data)
taxi_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 625134 entries, 0 to 625133
Data columns (total 28 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   id                  625134 non-null  object        
 1   vendor_id           625134 non-null  int64         
 2   pickup_datetime     625134 non-null  datetime64[ns]
 3   passenger_count     625134 non-null  int64         
 4   pickup_longitude    625134 non-null  float64       
 5   pickup_latitude     625134 non-null  float64       
 6   dropoff_longitude   625134 non-null  float64       
 7   dropoff_latitude    625134 non-null  float64       
 8   store_and_fwd_flag  625134 non-null  object        
 9   pickup_data         625134 non-null  datetime64[ns]
 10  pickup_hour         625134 non-null  int32         
 11  pickup_day_of_week  625134 non-null  object        
 12  day                 21882 non-null   object        
 13  date_x              21882 non

In [ ]:
print('Shape of data: {}'.format(taxi_data.shape))
print('Columns: {}'.format(taxi_data.columns))

Shape of data: (625134, 28)
Columns: Index(['id', 'vendor_id', 'pickup_datetime', 'passenger_count',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'store_and_fwd_flag', 'pickup_data', 'pickup_hour',
       'pickup_day_of_week', 'day', 'date_x', 'holiday', 'total_distance',
       'total_travel_time', 'number_of_steps', 'haversine_distance',
       'direction', 'geo_cluster', 'date_y', 'hour', 'temperature',
       'visibility', 'wind speed', 'precip', 'events'],
      dtype='object')


In [ ]:
taxi_data.drop(columns=['day', 'date_x', 'date_y', 'hour'], inplace=True)
taxi_data.shape

(625134, 24)

In [ ]:
train_data = taxi_data.copy()
train_data.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_data,...,total_travel_time,number_of_steps,haversine_distance,direction,geo_cluster,temperature,visibility,wind speed,precip,events
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N,2016-06-30,...,424.6,4,2.746426,-3.595224,1,24.4,16.1,0.0,0.0,None
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N,2016-06-30,...,200.0,4,2.759239,172.278835,6,24.4,16.1,0.0,0.0,None
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N,2016-06-30,...,193.2,4,1.306155,133.326248,6,24.4,16.1,0.0,0.0,None
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N,2016-06-30,...,494.8,11,5.269088,-150.956833,1,24.4,16.1,0.0,0.0,None
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N,2016-06-30,...,103.2,4,0.960842,130.260381,1,24.4,16.1,0.0,0.0,None


In [ ]:
train_data.columns

Index(['id', 'vendor_id', 'pickup_datetime', 'passenger_count',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'store_and_fwd_flag', 'pickup_data', 'pickup_hour',
       'pickup_day_of_week', 'holiday', 'total_distance', 'total_travel_time',
       'number_of_steps', 'haversine_distance', 'direction', 'geo_cluster',
       'temperature', 'visibility', 'wind speed', 'precip', 'events'],
      dtype='object')

In [ ]:
drop_columns = ['pickup_datetime', 'pickup_data']
train_data = train_data.drop(columns=drop_columns)
print('Shape of data:  {}'.format(train_data.shape))

Shape of data:  (625134, 22)


In [ ]:
train_data['vendor_id'] = train_data['vendor_id'].apply(lambda arg: 0 if arg == 1 else 1)

In [ ]:
train_data['store_and_fwd_flag'] = train_data['store_and_fwd_flag'].apply(lambda arg: 0 if arg == 'N' else 1)

In [ ]:
one_hot = taxi_data[['pickup_day_of_week', 'geo_cluster', 'events']]
one_hot.head()

,pickup_day_of_week,geo_cluster,events
0,Thursday,1,None
1,Thursday,6,None
2,Thursday,6,None
3,Thursday,1,None
4,Thursday,1,None


In [ ]:
one_hot_encoder = joblib.load('/content/drive/MyDrive/models/OHE_PP2.pkl')

In [ ]:
one_hot_encoder.categories_

[array(['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday',
        'Wednesday'], dtype=object),
 array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32),
 array(['Fog', 'None', 'Rain', 'Snow'], dtype=object)]

In [ ]:
data_one = one_hot_encoder.transform(one_hot)
column_names = one_hot_encoder.get_feature_names_out()
# data_one = pd.DataFrame(data=[data_one])
# data_one.head()
[data_one], column_names

([<625134x18 sparse matrix of type '<class 'numpy.float64'>'
  	with 1611476 stored elements in Compressed Sparse Row format>],
 array(['pickup_day_of_week_Monday', 'pickup_day_of_week_Saturday',
        'pickup_day_of_week_Sunday', 'pickup_day_of_week_Thursday',
        'pickup_day_of_week_Tuesday', 'pickup_day_of_week_Wednesday',
        'geo_cluster_1', 'geo_cluster_2', 'geo_cluster_3', 'geo_cluster_4',
        'geo_cluster_5', 'geo_cluster_6', 'geo_cluster_7', 'geo_cluster_8',
        'geo_cluster_9', 'events_None', 'events_Rain', 'events_Snow'],
       dtype=object))

In [ ]:
data_one = pd.DataFrame(data_one.toarray(), columns=column_names, dtype=np.int8)
data_one.head()

,pickup_day_of_week_Monday,pickup_day_of_week_Saturday,pickup_day_of_week_Sunday,pickup_day_of_week_Thursday,pickup_day_of_week_Tuesday,pickup_day_of_week_Wednesday,geo_cluster_1,geo_cluster_2,geo_cluster_3,geo_cluster_4,geo_cluster_5,geo_cluster_6,geo_cluster_7,geo_cluster_8,geo_cluster_9,events_None,events_Rain,events_Snow
0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0
2,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0
3,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0
4,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0


Добавим полученную таблицу с закодированными признаками:

In [ ]:
train_data = pd.concat(
    [train_data.reset_index(drop=True).drop(['pickup_day_of_week', 'geo_cluster', 'events'], axis=1), data_one],
    axis=1
)
print('Shape of data: {}'.format(train_data.shape))

Shape of data: (625134, 37)


In [ ]:
X = train_data.drop(['trip_duration', 'trip_duration_log'], axis=1)
y = train_data['trip_duration']
y_log = train_data['trip_duration_log']

In [ ]:
train_data.drop(columns=['id'], inplace=True)

In [ ]:
scaler = joblib.load('/content/drive/MyDrive/models/scaler.pkl')

In [ ]:
xtest = train_data.loc[:, ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag',
       'pickup_hour', 'holiday', 'total_distance', 'total_travel_time',
       'number_of_steps', 'haversine_distance', 'direction', 'temperature',
       'visibility', 'wind speed', 'precip', 'pickup_day_of_week_Monday',
       'pickup_day_of_week_Saturday', 'pickup_day_of_week_Sunday',
       'pickup_day_of_week_Thursday', 'pickup_day_of_week_Tuesday',
       'pickup_day_of_week_Wednesday', 'geo_cluster_1', 'geo_cluster_2',
       'geo_cluster_3', 'geo_cluster_4', 'geo_cluster_5', 'geo_cluster_6',
       'geo_cluster_7', 'geo_cluster_8', 'geo_cluster_9', 'events_None',
       'events_Rain', 'events_Snow']]
xtest = scaler.transform(xtest)

In [ ]:
test_data = pd.read_csv("/content/Project5_test_data.csv")
osrm_data_test = pd.read_csv("/content/Project5_osrm_data_test.csv")
test_id = test_data['id']

In [ ]:
test_data.head()

,id,starting_street,end_street,total_distance,total_travel_time,number_of_steps,street_for_each_step,distance_per_step,travel_time_per_step,step_maneuvers,step_direction,step_location_list
0,id0771704,6th Avenue,10th Avenue,1497.1,200.2,7,6th Avenue|West 17th Street|9th Avenue|9th Ave...,188.7|825.1|96.4|58.6|267.5|60.7|0,32.4|103.7|12.7|10.7|34|6.7|0,depart|turn|turn|continue|turn|turn|arrive,left|left|left|slight right|right|right|arrive,"-73.996527,40.737786|-73.995446,40.739272|-74...."
1,id3274209,5th Avenue,5th Avenue,1427.1,141.5,2,5th Avenue|5th Avenue,1427.1|0,141.5|0,depart|arrive,none|arrive,"-73.976974,40.75885|-73.98516,40.747618"
2,id2756455,East 18th Street,Park Avenue,2312.3,324.6,9,East 18th Street|Irving Place|East 19th Street...,19.3|74.1|148.9|632.1|163.9|1111.5|144.7|17.7|0,9.6|15.6|20.7|92.1|24.4|136.2|24.9|1.1|0,depart|turn|turn|turn|turn|turn|turn|turn|arrive,left|left|left|right|left|right|right|right|ar...,"-73.987161,40.736551|-73.986961,40.736466|-73...."
3,id3684027,Madison Avenue,6th Avenue,931.8,84.2,4,Madison Avenue|East 49th Street|6th Avenue|6th...,199.3|466.6|265.8|0,24.4|36.5|23.3|0,depart|turn|turn|arrive,right|left|right|arrive,"-73.977191,40.755664|-73.976045,40.757232|-73...."
4,id3101285,Madison Avenue,West 83rd Street,2501.7,294.7,8,Madison Avenue|East 91st Street|5th Avenue|86t...,31|154.9|491|907.6|272.1|248|397.2|0,9.4|29.7|46|103.6|25.1|21.8|59.1|0,depart|turn|turn|turn|new name|turn|turn|arrive,right|left|left|right|straight|left|right|arrive,"-73.956667,40.783797|-73.956489,40.784041|-73...."


Перед созданием прогноза для тестовой выборки необходимо произвести все манипуляции с данными, которые мы производили с тренировочной выборкой, а именно:
* Перевести признак pickup_datetime в формат datetime;
* Добавить новые признаки (временные, географические, погодные и другие факторы);
* Произвести очистку данных от пропусков;
* Произвести кодировку категориальных признаков:
    * Закодировать бинарные признаки;
    * Закодировать номинальные признаки с помощью обученного на тренировочной выборке OneHotEncoder’а;
* Сформировать матрицу наблюдений, оставив в таблице только те признаки, которые были отобраны с помощью SelectKBest;
* Нормализовать данные с помощью обученного на тренировочной выборке MinMaxScaler’а.


Только после выполнения всех этих шагов можно сделать предсказание длительности поездки для тестовой выборки. Не забудьте перевести предсказания из логарифмического масштаба в истинный, используя формулу:
$$y_i=exp(z_i)-1$$

После того, как вы сформируете предсказание длительности поездок на тестовой выборке вам необходимо будет создать submission-файл в формате csv, отправить его на платформу Kaggle и посмотреть на результирующее значение метрики RMSLE на тестовой выборке.

Код для создания submission-файла:


In [ ]:
model = joblib.load('/content/drive/MyDrive/models/boost.pkl')
pred = model.predict(xtest)

In [ ]:
pred = np.exp(pred) - 1

In [ ]:
# ваш код здесь
submission = pd.DataFrame({'id': test_id.values, 'trip_duration': pred})
submission.to_csv('/content/submission_gb.csv', index=False)